# CSV Preparation Update

In [1]:
import pandas as pd
import requests

# Minimum working request
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 51.0447,
    "longitude": -114.0719,
    "start_date": "2025-01-01",
    "end_date": "2025-01-02",
    "hourly": "temperature_2m,precipitation",
    "models": "era5",
    "timezone": "GMT",
    "timeformat": "unixtime",
}

response = requests.get(
    url,
    params=params,
    timeout=60,
)
response.raise_for_status()

df = pd.DataFrame(
    response.json()["hourly"]
)

df["time"] = pd.to_datetime(
    df["time"],
    unit="s",
    utc=True,
)

df.head()


,time,temperature_2m,precipitation
0,2025-01-01 00:00:00+00:00,-7.7,0.1
1,2025-01-01 01:00:00+00:00,-8.3,0.1
2,2025-01-01 02:00:00+00:00,-8.9,0.1
3,2025-01-01 03:00:00+00:00,-9.1,0.2
4,2025-01-01 04:00:00+00:00,-9.4,0.2


In [2]:
df.dtypes


time              datetime64[s, UTC]
temperature_2m               float64
precipitation                float64
dtype: object

In [3]:
from pathlib import Path

# CSV stores time as text; parsing restores datetime semantics.
sample_path = Path("sample_weather.csv")

df.to_csv(
    sample_path,
    index=False,
)

print(
    pd.read_csv(sample_path).dtypes
)

print(
    pd.read_csv(
        sample_path,
        parse_dates=["time"],
    ).dtypes
)

sample_path.unlink()

time                  str
temperature_2m    float64
precipitation     float64
dtype: object
time              datetime64[us, UTC]
temperature_2m                float64
precipitation                 float64
dtype: object


In [4]:
import time

# Configuration
URL = "https://archive-api.open-meteo.com/v1/archive"

LATITUDE = 51.0447
LONGITUDE = -114.0719
MODEL = "era5"
LOCAL_TIMEZONE = "America/Edmonton"

HOURLY_VARIABLES = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "precipitation",
    "rain",
    "snowfall",
    "weather_code",
    "cloud_cover",
    "pressure_msl",
    "surface_pressure",
    "wind_speed_10m",
    "wind_direction_10m",
    "shortwave_radiation",
    "vapour_pressure_deficit",
    "soil_temperature_0_to_7cm",
]

START_LOCAL = pd.Timestamp(
    "2000-01-01 00:00",
    tz=LOCAL_TIMEZONE,
)

END_LOCAL = pd.Timestamp(
    "2026-10-01 00:00",
    tz=LOCAL_TIMEZONE,
)

REQUEST_DELAY_SECONDS = 4

DATA_PATH = Path("data") / "weather.csv"


In [5]:
# First pass at local calendar-year batches
batches = []
batch_start = START_LOCAL

while batch_start < END_LOCAL:
    next_year = pd.Timestamp(
        year=batch_start.year + 1,
        month=1,
        day=1,
        tz=batch_start.tz,
    )

    batch_end = min(
        next_year,
        END_LOCAL,
    )

    batches.append(
        (batch_start, batch_end)
    )

    batch_start = batch_end

batches[:2], batches[-2:]


([(Timestamp('2000-01-01 00:00:00-0700', tz='America/Edmonton'),
   Timestamp('2001-01-01 00:00:00-0700', tz='America/Edmonton')),
  (Timestamp('2001-01-01 00:00:00-0700', tz='America/Edmonton'),
   Timestamp('2002-01-01 00:00:00-0700', tz='America/Edmonton'))],
 [(Timestamp('2025-01-01 00:00:00-0700', tz='America/Edmonton'),
   Timestamp('2026-01-01 00:00:00-0700', tz='America/Edmonton')),
  (Timestamp('2026-01-01 00:00:00-0700', tz='America/Edmonton'),
   Timestamp('2026-10-01 00:00:00-0600', tz='America/Edmonton'))])

In [6]:
def create_year_batches(
    start_local,
    end_local,
):
    batches = []
    batch_start = start_local

    while batch_start < end_local:
        next_year = pd.Timestamp(
            year=batch_start.year + 1,
            month=1,
            day=1,
            tz=batch_start.tz,
        )

        batch_end = min(
            next_year,
            end_local,
        )

        batches.append(
            (batch_start, batch_end)
        )

        batch_start = batch_end

    return batches


In [7]:
def fetch_weather_batch(
    start_local,
    end_local,
):
    start_utc = start_local.tz_convert("UTC")
    end_utc = end_local.tz_convert("UTC")

    # API accepts dates, so request the UTC envelope then trim.
    params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "start_date": start_utc.date().isoformat(),
        "end_date": (
            end_utc
            - pd.Timedelta(hours=1)
        ).date().isoformat(),
        "hourly": ",".join(
            HOURLY_VARIABLES
        ),
        "models": MODEL,
        "timezone": "GMT",
        "timeformat": "unixtime",
    }

    response = requests.get(
        URL,
        params=params,
        timeout=60,
    )
    response.raise_for_status()

    batch_df = pd.DataFrame(
        response.json()["hourly"]
    )

    batch_df["time"] = pd.to_datetime(
        batch_df["time"],
        unit="s",
        utc=True,
    )

    return (
        batch_df.loc[
            batch_df["time"].ge(
                start_utc
            )
            & batch_df["time"].lt(
                end_utc
            ),
            [
                "time",
                *HOURLY_VARIABLES,
            ],
        ]
        .reset_index(drop=True)
    )


In [8]:
# Small function check
test_df = fetch_weather_batch(
    pd.Timestamp(
        "2025-01-01 00:00",
        tz=LOCAL_TIMEZONE,
    ),
    pd.Timestamp(
        "2025-01-03 00:00",
        tz=LOCAL_TIMEZONE,
    ),
)

test_df.head()


,time,temperature_2m,relative_humidity_2m,dew_point_2m,precipitation,rain,snowfall,weather_code,cloud_cover,pressure_msl,surface_pressure,wind_speed_10m,wind_direction_10m,shortwave_radiation,vapour_pressure_deficit,soil_temperature_0_to_7cm
0,2025-01-01 07:00:00+00:00,-9.8,75,-13.4,0.1,0.0,0.07,71,100,1030.0,900.3,7.0,3,0.0,0.07,-5.4
1,2025-01-01 08:00:00+00:00,-9.8,74,-13.6,0.1,0.0,0.07,71,100,1029.8,900.1,6.1,5,0.0,0.08,-5.9
2,2025-01-01 09:00:00+00:00,-10.1,73,-13.9,0.1,0.0,0.07,71,99,1029.6,899.8,5.3,10,0.0,0.08,-6.4
3,2025-01-01 10:00:00+00:00,-11.5,77,-14.8,0.1,0.0,0.07,71,98,1029.9,899.4,6.5,6,0.0,0.06,-5.7
4,2025-01-01 11:00:00+00:00,-11.2,76,-14.6,0.1,0.0,0.07,71,99,1029.9,899.6,6.4,10,0.0,0.06,-6.0


In [9]:
def fetch_weather(
    start_local,
    end_local,
):
    batches = create_year_batches(
        start_local,
        end_local,
    )

    frames = []

    for i, (
        batch_start,
        batch_end,
    ) in enumerate(
        batches,
        start=1,
    ):
        print(
            f"Request {i}/{len(batches)}: "
            f"{batch_start} -> {batch_end}"
        )

        frames.append(
            fetch_weather_batch(
                batch_start,
                batch_end,
            )
        )

        if i < len(batches):
            time.sleep(
                REQUEST_DELAY_SECONDS
            )

    df = pd.concat(
        frames,
        ignore_index=True,
    )

    # Catch gaps, overlaps, boundary errors, and unexpected hours.
    expected_time = pd.date_range(
        start=start_local.tz_convert(
            "UTC"
        ),
        end=end_local.tz_convert(
            "UTC"
        ),
        freq="h",
        inclusive="left",
    )

    if not pd.DatetimeIndex(
        df["time"]
    ).equals(expected_time):
        raise ValueError(
            "Unexpected hourly coverage."
        )

    return df


In [10]:
# Full preparation
df = fetch_weather(
    START_LOCAL,
    END_LOCAL,
)

DATA_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

df.to_csv(
    DATA_PATH,
    index=False,
)

print(f"Rows: {len(df):,}")
print(
    f"Range: {df['time'].min()} "
    f"to {df['time'].max()}"
)
print(f"Saved: {DATA_PATH}")


Request 1/27: 2000-01-01 00:00:00-07:00 -> 2001-01-01 00:00:00-07:00
Request 2/27: 2001-01-01 00:00:00-07:00 -> 2002-01-01 00:00:00-07:00
Request 3/27: 2002-01-01 00:00:00-07:00 -> 2003-01-01 00:00:00-07:00
Request 4/27: 2003-01-01 00:00:00-07:00 -> 2004-01-01 00:00:00-07:00
Request 5/27: 2004-01-01 00:00:00-07:00 -> 2005-01-01 00:00:00-07:00
Request 6/27: 2005-01-01 00:00:00-07:00 -> 2006-01-01 00:00:00-07:00
Request 7/27: 2006-01-01 00:00:00-07:00 -> 2007-01-01 00:00:00-07:00
Request 8/27: 2007-01-01 00:00:00-07:00 -> 2008-01-01 00:00:00-07:00
Request 9/27: 2008-01-01 00:00:00-07:00 -> 2009-01-01 00:00:00-07:00
Request 10/27: 2009-01-01 00:00:00-07:00 -> 2010-01-01 00:00:00-07:00
Request 11/27: 2010-01-01 00:00:00-07:00 -> 2011-01-01 00:00:00-07:00
Request 12/27: 2011-01-01 00:00:00-07:00 -> 2012-01-01 00:00:00-07:00
Request 13/27: 2012-01-01 00:00:00-07:00 -> 2013-01-01 00:00:00-07:00
Request 14/27: 2013-01-01 00:00:00-07:00 -> 2014-01-01 00:00:00-07:00
Request 15/27: 2014-01-01 00:

In [11]:
def main():
    df = fetch_weather(
        START_LOCAL,
        END_LOCAL,
    )

    DATA_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    df.to_csv(
        DATA_PATH,
        index=False,
    )

    print(f"Rows: {len(df):,}")
    print(
        f"Range: {df['time'].min()} "
        f"to {df['time'].max()}"
    )
    print(f"Saved: {DATA_PATH}")
